<table>
  <tr>
    <td><p style="font-size:45px; color: #55BBD2">Analysis of light microscopy images in Python </p></td>
    <td><img src="../ressources/lmb_logo.svg" alt="LMB Logo" width="500" height="600" align="right"></td>
  </tr>
</table>
<table>
  <tr>
    <td><p style="font-size:15px; color: #55BBD2">Version: October 2026</p></td>
  </tr>
</table>

# Question of interest: Stress granule quantification at single cell level

<b>Dataset:</b> Stress_granules_positions.lsm

<b>Credit:</b> Lucas Albacete 

| **Step** | **What it teaches** | **Python packages / functions** |
| --- | --- | --- |
| **1. Inspect the image** | Dimensions, channels and physical units | `pathlib.Path`, `bioio.BioImage`, `xarray.isel`, `matplotlib.pyplot` , `napari.Viewer`,`napari.add_image`|
| **2. Estimate background** | Background intensity and variation; optional background subtraction using white top-hat | `numpy.mean`, `numpy.std`, `skimage.morphology.white_tophat`, `disk` |
| **3. Segment granules** | Calculate an intensity threshold, create a binary mask and label connected regions | `scipy.stats.norm.isf`, NumPy comparisons, `skimage.measure.label`, `napari.add_labels`|
| **4. Refine granule masks** | Remove small objects and border objects; close small gaps; optionally split touching granules (not included) | `skimage.morphology.remove_small_objects`, `closing`; `skimage.segmentation.clear_border`|
| **5. Measure** | Granule sizes and intensity| `skimage.measure.regionprops_table`|
| **6. Identify cells** | Create or load a label image where each cell has a unique integer ID | `models.CellposeModel`, `model.eval` |
| **7. Assign granules to cells** | Relate each granule to a cell; identify granules outside or crossing cell boundaries; per-cell summaries | NumPy indexing, `skimage.measure.regionprops` , `pandas.DataFrame.groupby`|
| **8. Validate and compare** | Check masks and overlays; compare measurements between cells or conditions | `napari.add_points`|
| **9. Save data to CSV** | Export measurement tables for further analysis | `pandas.DataFrame.to_csv`|

## 1. Inspect the image
### Open an image with BioIO

Create a `BioImage` object. For files containing multiple scenes, the first scene is selected by default. [BioIO documentation](https://bioio-devs.github.io/bioio/OVERVIEW.html)

In [ ]:
from pathlib import Path
from bioio import BioImage

#  give the path to the data of interest
data_folder = Path('../data')

#  load the image and the metadata into the varible "image"
image = BioImage(data_folder / 'Stress_granules_positions.tif')
print(type(image))
print(image.dtype)

### Inspect metadata

Metadata describes the image and how it was acquired. Depending on the file format and reader, it may include channel names, emission wavelengths, voxel sizes, and other acquisition parameters.


| Attribute | Description |
|---|---|
| `image.channel_names` | Channel names |
| `image.physical_pixel_sizes` | Physical voxel sizes in micrometres (Z, Y, X) |
| `image.physical_pixel_sizes.X` | Physical pixel size along X in micrometres |
| `image.dims` | Dimensions object |
| `image.dims.order` | Dimension order, typically `"TCZYX"` |
| `image.dims.X` | Number of pixels along X |
| `image.shape` | Image shape in dimension order |

In [ ]:
#  show example metadata of the image
print(f'The image dimension are {image.dims}')
print(f'The pixel size in X is {image.physical_pixel_sizes}')
print(f'The pixel size in X is {image.physical_pixel_sizes.X:.4f} microns')

### Image is data

A digital image is a finite numeric representation of the intensity values. It is composed of picture elements known as pixels. 

In Python, with BioIO, use `.data` for a NumPy array (typically ordered (T, C, Z, Y, X)) or `.xarray_data` for an xarray `DataArray` with named dimensions. Both load the current scene into memory.

In [ ]:
# xarray DataArray with named dimensions; returns 5D TCZYX xarray data array backed by numpy
data = image.xarray_data
print(type(data))
print(data.dtype)

Select a single 2D plane using dimension names. <b>Indices start at zero.<b>

In [ ]:
# First time point, second channel, first Z plane
plane = data.isel(T=0, C=1, Z=0)
print(type(plane))
print(plane.dtype)
print(plane.sizes)

Alternatively, use `image.get_image_data()` to request a NumPy array in a specified dimension order.

In [ ]:
# All channels and Z planes at the first time point
# Output dimension order: (C, Z, Y, X)
stack = image.get_image_data("CZYX", T=0)
print(type(stack))
print(stack.dtype)
print(stack.shape)

### Visualize with matplotlib
Display an image. <b>Indices start at zero.<b>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select the first time point, second channel, and first Z plane (0-based indices)
plane = data.isel(T=0, C=1, Z=0)

# Calculate the 1st and 99th percentile intensities to set the display contrast
vmin, vmax = np.percentile(plane, (1, 99))

# Display the plane in grayscale using these contrast limits
plt.imshow(plane, cmap="gray", vmin=vmin, vmax=vmax)

Now let's display all the channels of the image in a subplot using a `for` loop:

In [ ]:
# Get the number of channels using the named channel dimension
num_channels = data.sizes["C"]

# Create one subplot per channel
fig, axes = plt.subplots(1, num_channels, figsize=(4 * num_channels, 4))

# Ensure axes is iterable even when there is only one channel
axes = np.atleast_1d(axes)

# Loop over each channel index and its corresponding subplot
for k, ax in enumerate(axes):
    # Select the first time point and first Z plane for this channel
    plane = data.isel(T=0, C=k, Z=0)

    # Calculate contrast limits separately for each channel
    vmin, vmax = np.percentile(plane, (1, 99))

    # Display the plane using the "hot" colormap and calculated contrast limits
    ax.imshow(plane, cmap="hot", vmin=vmin, vmax=vmax)

    # Hide the axes and ticks
    ax.set_axis_off()

    # Use the channel name as the subplot title
    ax.set_title(image.channel_names[k])

# Adjust spacing to prevent subplot overlap
plt.tight_layout()

# Show the figure
plt.show()

In the following cell, we crop a part of the image, display it in a figure and overlay the pixel values.

In [ ]:
# Select time point 0, channel 1 (the second channel), and z-plane 0.
# Take every fifth pixel within a 50 × 50 pixel region.
# The resulting array contains 10 × 10 sampled pixels.
crop = data.isel(
    T=0,
    C=1,
    Z=0,
    Y=np.arange(1326, 1376, 5),  # Row indices; 1376 is excluded
    X=np.arange(1423, 1473, 5),  # Column indices; 1473 is excluded
)

# Create a figure and display the sampled image in grayscale.
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(crop, cmap="gray")

# Extract the pixel values as a NumPy array.
values = crop.values

# Loop through every row (i) and column (j).
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        # Choose the text colour using an intensity threshold of 5.
        c = "white" if values[i, j] < 5 else "black"

        # Display the intensity at the pixel centre.
        # Matplotlib uses (x, y) coordinates, so pass (j, i).
        # :.0f formats the value with zero decimal places.
        ax.text(
            j, i, f"{values[i, j]:.0f}",
            color=c, ha="center", va="center", size=10,
        )

# Hide the axes, add a title, and display the figure.
ax.axis("off")
ax.set_title("Pixel values")
plt.show()

(optional) Let say we would like to visualize the profile of the intensity distribution of a given channel along the x-axis at the mid-position of the y-axis in the image. 

In [ ]:
# Select channel 1 (the second channel; indexing starts at 0).
k = 1

# Extract the intensity along row Y=956 at time point 0 and z-plane 0.
# X remains unselected, giving a one-dimensional intensity profile.
intensity = data.isel(T=0, C=k, Z=0, Y=956)

# Create an x-coordinate for each pixel along the row.
x = np.arange(len(intensity))

# Plot intensity against horizontal pixel position.
plt.plot(x, intensity)

# Label the axes; a.u. means arbitrary units.
plt.xlabel("x [pixels]")
plt.ylabel("Intensity [a.u.]")

# Display the channel number using counting that starts at 1.
plt.title(f"Intensity for channel {k + 1}")

### Display with napari viewer (interactive)

Napari is an interactive Python tool for viewing scientific images. It displays data as `layers`, allowing  2D, 3D, and higher-dimensional images, segmentation masks, and detected points to be overlaid in the same viewer, and seamlessly integrate exploration with computation and annotation.

| Function | What it shows | Input data | Useful parameters |
|---|---|---|---|
| `viewer.add_image()` | Intensity images: multichannel images, Z-stacks, and time series | An image array, such as `(Y, X)` or `(Z, Y, X)` | `name`: layer name; `colormap`: colour scheme, e.g. `"gray"`; `contrast_limits`: displayed intensity range, e.g. `[0, 1000]`; `channel_axis`: axis containing channels, split into separate layers; `blending`: how layers overlap, e.g. `"additive"` |
| `viewer.add_labels()` | Segmentation masks, with different colours for different objects | An integer array: `0` = background; positive integers = object IDs | `name`: layer name; `opacity`: transparency from `0` to `1`; `scale`: pixel or voxel spacing; `contour`: boundary thickness in pixels (`0` displays filled objects) |
| `viewer.add_points()` | Coordinates of detected spots, cell centres, or manually marked locations | An `(N, D)` array: `N` points in `D` dimensions, e.g. `(Y, X)` or `(Z, Y, X)` | `name`: layer name; `size`: marker size; `face_color`: marker fill colour; `symbol`: marker shape, e.g. `"disc"` or `"cross"`; `features`: measurements associated with each point; `text`: displayed annotations |

These parameters follow the [Napari viewer API and examples](https://napari.org/stable/api/napari.Viewer.html).
Coordinate order matters: points follow the array’s axis order—usually `(Y, X)` for 2D images, rather than Matplotlib’s `(X, Y)`. Apply the same `scale` to corresponding image, labels, and points layers to keep them aligned.

In [ ]:
import napari

# Open an interactive napari viewer
viewer = napari.Viewer()

# add the images, one channel per layer
viewer.add_image(
    data.values,
    channel_axis=data.get_axis_num("C"),
    name=image.channel_names,
)

# Start the event loop to keep the viewer open
napari.run()

## 2. Detect stress granules
### Estimate and subtract background
`Top-hat` filtering is a morphological image-processing method that highlights small features relative to their surroundings. For bright granules, a white top-hat estimates the background using an operation called opening, then subtracts it from the original image.

The `disk` size controls the scale of features retained in the result. Choose a disk larger than the granules you want to detect. For example, if the largest granules are approximately 40 pixels in diameter, a disk with a radius of 30 pixels is a starting point. Compare the original image, estimated background and filtered result, and adjust the radius if granule signal remains in the background estimate or larger structures are retained.

In [ ]:
from skimage import morphology

# Select the stress-granule channel (C=1) at the first time point and Z-plane
plane = data.isel(T=0, C=1, Z=0)

# Create a disk-shaped footprint with a radius of 30 pixels
footprint = morphology.disk(30)

# Apply white top-hat filtering to highlight small bright features
# relative to the surrounding background
res = morphology.white_tophat(plane, footprint=footprint)

# Open a napari viewer
viewer = napari.Viewer()

# Add the original plane and top-hat result as separate image layers
viewer.add_image(plane, name="Original image")
viewer.add_image(res, name="White top-hat result")

napari.run()

### Threshold and label candidate granules

The mean and standard deviation of the image intensities are used to calculate a threshold:

$$
T = \mu + k\sigma
$$

For a one-sided probability of $p = 0.001$, $k \approx 3.09$, assuming normally distributed background intensities. Pixels above the threshold are classified as candidate signal, and connected foreground pixels are assigned unique labels.

<img src="../ressources/intensity_threshold.png" alt="drawing" width="800"/>

In [ ]:
import numpy as np
from scipy.stats import norm
from skimage import measure

# Select the stress-granule channel and one image plane
stress_g = data.isel(T=0, C=1, Z=0)
res = np.asarray(stress_g, dtype=np.float32)

# Estimate background mean and standard deviation
mean_bg = np.mean(res)
sigma_bg = np.std(res, ddof=1)

# One-sided probability of a background pixel exceeding the threshold
p = 0.001
k = norm.isf(p)

# Calculate the intensity threshold
threshold = mean_bg + k * sigma_bg

# Classify pixels above the threshold as candidate signal
signal_mask = res > threshold

print(f"Background mean: {mean_bg:.2f}")
print(f"Background sigma: {sigma_bg:.2f}")
print(f"Coefficient: {k:.2f}")
print(f"Threshold: {threshold:.2f}")

# Label connected foreground regions
granule_labels = measure.label(signal_mask)

Visualize the image and the labels; toggle the labels layer to compare the segmentation with the image and check for missed or incorrectly segmented granules.

In [ ]:
viewer = napari.Viewer()

# Display the stress-granule image as an image layer
viewer.add_image(res, name="Stress granules")

# Overlay the segmentation as a labels layer
# Each granule has a unique integer ID; 0 represents background
viewer.add_labels(granule_labels, name="Granule labels")

napari.run()

### Post processing of binary masks
After thresholding, you can remove small regions, optionally close small gaps, and label the remaining granules, which can be done using morphological operations.
In the morphological operations, <br>
- **Erosion:** Shrinks foreground objects and may remove small objects.
- **Dilation:** Expands foreground objects and may fill small gaps or holes.
- **Closing:** Dilation followed by erosion; closes small gaps and holes while largely preserving object size.

In [ ]:
import matplotlib.pyplot as plt
from skimage import morphology, segmentation, measure, exposure, color

# Remove small foreground regions, such as isolated noise
min_area = 10  # Minimum region size in pixels
only_large_objects = morphology.remove_small_objects(
    signal_mask,
    min_size=min_area,
)

# Optionally close small gaps within granules
# Use a small footprint to avoid joining nearby granules
closed = morphology.closing(
    only_large_objects,
    morphology.disk(1),
)

# Remove granules touching the image border
cleared = segmentation.clear_border(closed)

# Assign a unique integer label to each connected region
granule_labels = measure.label(cleared)

# Overlay the labelled granules on the original image
vmin, vmax = np.percentile(res, (1, 99))
# Rescale a copy to 0–1 for the overlay
img_display = exposure.rescale_intensity(
    res,
    in_range=(vmin, vmax),
    out_range=(0.0, 1.0),
)
granule_overlay = color.label2rgb(
    granule_labels,
    image=img_display,
    bg_label=0,
)

print(f"Number of granules: {granule_labels.max()}")

# Display each processing step
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].imshow(res, cmap="gray")
axes[0, 0].set_title("Original image")

axes[0, 1].imshow(signal_mask, cmap="gray")
axes[0, 1].set_title("Threshold mask")

axes[0, 2].imshow(only_large_objects, cmap="gray")
axes[0, 2].set_title("Small regions removed")

axes[1, 0].imshow(cleared, cmap="gray")
axes[1, 0].set_title("Closed and border cleared")

axes[1, 1].imshow(granule_labels, cmap="nipy_spectral")
axes[1, 1].set_title("Label image")

axes[1, 2].imshow(granule_overlay)
axes[1, 2].set_title("Granule overlay")

for ax in axes.flat:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

Adjust `min_area` to suit the smallest real granules. Closing can merge nearby granules, so inspect the result and skip that step if it is unnecessary. Likewise, `clear_border` removes entire granules touching the edge; omit it if you want to retain partial objects. Check with Napari again.

In [ ]:
viewer = napari.Viewer()

viewer.add_image(res, name="Stress granules")
viewer.add_labels(granule_labels, name="Granule labels")

napari.run()

### Watershed???

## 3. Measurement

Measure granule area and intensity using the labelled masks. Use the original, unsmoothed image for raw intensity measurements, or an unsmoothed, background-corrected image to measure intensity above background.

[skimage.measure.regionprops_table()](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops_table) measures properties of labelled objects and returns them as a dictionary of columns, which can be converted into a pandas DataFrame for downstream analysis.

| Parameter | Type | Description |
| --- | --- | --- |
| `label_image` | ndarray | Label image; regions labelled 0 are excluded. |
| `intensity_image` | ndarray, optional | Input intensity image matching the label image’s shape. An optional channel dimension must be the last axis. |
| `properties` | tuple or list of str, optional | Properties that will be included in the resulting dictionary, such as `("label", "area", "centroid", "intensity_mean")`. For a list of available properties, please see [regionprops()](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops). Users should remember to add “label” to keep track of region identities. |
| `spacing` | tuple of float, optional | Pixel spacing for each image axis. |

The label image defines **where to measure**, while `intensity_image` provides **the intensity values**.

**Returns:** A dictionary containing an array for each property, suitable for conversion to a pandas DataFrame.

In [ ]:
import pandas as pd

# Measure individual granules
granule_table = pd.DataFrame(
    measure.regionprops_table(
        granule_labels,
        intensity_image=res,
        properties=(
            "label",
            "area",
            "centroid",
            "intensity_mean",
        ),
    )
)

# Give columns descriptive names
granule_table = granule_table.rename(columns={
    "label": "granule_id",
    "area": "area_px",
    "centroid-0": "y",
    "centroid-1": "x",
})

granule_table.head

## 4. Segment the cell using CellPose (Machine Learning approach)

Image segmentation is the process of partitioning an image into multiple distinct regions. In semantic segmentation, regions can represent objects of interests against background. They can then be represented as a binary mask. Several type of objects can be further encoded using a label image. Finally, instance segmentation aims at distinguishing different identities of objects of potentially the same class and can also be represented using a label image.

[Cellpose](https://cellpose.readthedocs.io/en/latest/) is a generalist cellular segmentation algorithm, trained on diverse datasets that works on wide a range of data types.

Key features: 
- Instance segmentation
- Easy-to-use GUI
- A “zoo” of pre-trained models
- Well-documented Python API
- Simple fine-tuning of models via Human-in-the-Loop retraining


<img src="../ressources/cellpose_w.png" alt="drawing" width="800"/>


### Steps
- Get the pretrained model using `models.CellposeModel`
- Apply the model for segmentation using `model.eval`
  - Parameters:
    - img : np.ndarray, input image to segment.
    - channels (`indexing from 1`) : list of int, specifies which channels to use from the image:
      - channels[0]: channel to segment (e.g. cytoplasmic)
      - channels[1]: optional auxiliary channel (e.g. nuclear)
    - diameter : float or int, a user-defined cell diameter (in pixels) or an estimated one when set "diameter = None" or "diameter = 0". It is used to rescale the image internally to match the training conditions of the model. 
    - cellprob_threshold : float, threshold applied to the internal "cell probability" map output by the model. Higher values result in stricter segmentation. Typical range: 0.0 to 1.0
  - Returns:
    - mask : np.ndarray, segmentation mask with labeled regions.
    - flows : list of np.ndarray, list of flow-related outputs used in Cellpose for segmentation and visualization.
      - flows[0]: HSV-encoded XY flow image (H, W, 3), for visualization only.
      - flows[1]: Raw XY flow vectors (2, H, W), directing pixel movement toward cell centers.
      - flows[2]: Cell probability map (H, W); used to threshold foreground pixels.
      - flows[3]: Final pixel coordinates after Euler integration (2, H, W); used to define cell masks.
    - diams : float, estimated object diameter based on image content (may differ from input diameter).


### Check for GPU Availability
Checks for a CUDA-compatible GPU, which significantly accelerates deep learning model training and inference.

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

use_gpu = torch.cuda.is_available()

In [ ]:
from cellpose import models
import matplotlib.pyplot as plt

# Select a 2D multichannel image: (C, Y, X)
plane = data.isel(T=0, Z=0)

# Cellpose v3: pretrained cyto2 model
model = models.CellposeModel(gpu=use_gpu, model_type="cyto2")

# Cellpose channel numbers: 1 = nuclei channel, 3 = membrane channel
cells, flows, styles = model.eval(
    plane.values,
    channel_axis=0,
    channels=[4, 3],
    diameter=400,
    cellprob_threshold=1.0,
)

# Display channels and segmentation results
num_channels = plane.sizes["C"]
fig, axes = plt.subplots(
    1, num_channels + 3,
    figsize=(3 * (num_channels + 3), 4),
)

for k in range(num_channels):
    axes[k].imshow(plane.isel(C=k), cmap="gray")
    axes[k].set_axis_off()
    axes[k].set_title(image.channel_names[k])

axes[num_channels].imshow(cells, cmap="nipy_spectral", interpolation="nearest")
axes[num_channels].set_title("Labels")

axes[num_channels + 1].imshow(flows[0])
axes[num_channels + 1].set_title("Flows")

axes[num_channels + 2].imshow(flows[2], cmap="gray")
axes[num_channels + 2].set_title("Cell probability")

for ax in axes:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## 5. Assign granules to cells (Parenting)
Keep only granules entirely contained within one cell. Exclude granules outside cells or crossing cell boundaries.

In [ ]:
# Create an empty dictionary to map each granule ID to its cell ID
# Example: {granule_id: cell_id} → {1: 3, 2: 3, 5: 7}
granule_to_cell = {}

# Create an empty list to store rejected granule IDs
# Example: [4, 6, 9]
rejected = []

# Loop through each labelled granule
for region in measure.regionprops(granule_labels):

    # Get the (y, x) coordinates of all pixels in this granule
    yy, xx = region.coords.T

    # Find the unique cell labels beneath the granule
    # Cell label 0 represents background outside cells
    overlapping_cells = np.unique(cells[yy, xx])

    # Accept the granule only if all its pixels lie inside one cell
    if len(overlapping_cells) == 1 and overlapping_cells[0] > 0:
        # Add the granule ID and its cell ID to the dictionary
        granule_to_cell[region.label] = int(overlapping_cells[0])
    else:
        # Add the rejected granule ID to the list
        rejected.append(region.label)

# Set rejected granule pixels to 0; accepted label IDs stay unchanged
granule_labels[np.isin(granule_labels, rejected)] = 0

print(f"Accepted granules: {len(granule_to_cell)}")
print(f"Excluded granules: {len(rejected)}")

# Add the assigned cell ID
granule_table["cell_id"] = (
    granule_table["granule_id"]
    .map(granule_to_cell)
    .astype(int)
)

# Convert pixel area to physical area
PIXEL_SIZE_UM = image.physical_pixel_sizes
granule_table["area_um2"] = (
    granule_table["area_px"] * PIXEL_SIZE_UM**2
)

# Sum of pixel intensities in each granule
# granule_table["integrated_intensity"] = (
#     granule_table["mean_intensity"] * granule_table["area_px"]
# )

granule_table.head()

Create the cell table separately so that cells with zero granules are retained.

In [ ]:
# Measure every cell
cell_table = pd.DataFrame(
    measure.regionprops_table(
        cells,
        properties=("label", "area"),
    )
).rename(columns={
    "label": "cell_id",
    "area": "cell_area_px",
})

cell_table["cell_area_um2"] = (
    cell_table["cell_area_px"] * PIXEL_SIZE_UM**2
)

# Summarise granules belonging to each cell
summary = granule_table.groupby("cell_id").agg(
    granule_count=("granule_id", "size"),
    total_granule_area_um2=("area_um2", "sum"),
    mean_granule_area_um2=("area_um2", "mean"),
    total_granule_intensity=("integrated_intensity", "sum"),
)

# Keep all cells, including those without granules
cell_table = cell_table.merge(
    summary,
    on="cell_id",
    how="left",
)

# Counts and totals are zero for cells without granules
zero_columns = [
    "granule_count",
    "total_granule_area_um2",
    "total_granule_intensity",
]
cell_table[zero_columns] = cell_table[zero_columns].fillna(0)
cell_table["granule_count"] = cell_table["granule_count"].astype(int)

# Mean granule size stays NaN when a cell contains no granules
cell_table["pct_cell_covered"] = (
    100
    * cell_table["total_granule_area_um2"]
    / cell_table["cell_area_um2"]
)

# Here, a positive cell means one or more accepted granules
cell_table["granule_positive"] = cell_table["granule_count"] > 0

cell_table.head()

## 6. Validate 
Check several cells with different brightness levels. Look for missed dim granules, noise classified as granules, merged granules, and inaccurate cell boundaries.
Overlay the image, cell boundaries, and granule masks in napari. 

In [ ]:
v = napari.Viewer()

v.add_image(
    plane,
    name="Original image",
    colormap="gray",
)

v.add_image(
    res,
    name="Background corrected",
    colormap="gray",
    visible=False,
)

v.add_labels(
    cells,
    name="Cells",
    visible=False,
)

v.add_labels(
    granule_labels,
    name="Granules",
    opacity=0.6,
)

# Get granule IDs and centroid coordinates
props = measure.regionprops_table(
    granule_labels,
    properties=("label", "centroid"),
)

# Match each granule to its measured area
areas = (
    granule_table.set_index("granule_id")["area_um2"]
    .reindex(props["label"])
    .to_numpy()
)

# Display area labels at granule centres
v.add_points(
    np.column_stack((props["centroid-0"], props["centroid-1"])),
    name="Granule area (µm²)",
    size=1,
    face_color="transparent",
    border_color="transparent",
    features={"area_um2": areas},
    text={
        "string": "{area_um2:.2f} µm²",
        "color": "yellow",
        "size": 10,
        "translation": [0, 5],
    },
)

napari.run()

## 9. Save the analysis result

In [ ]:
import pandas as pd
from pathlib import Path

# Convert the granule-to-cell mapping into a table
assignments = pd.DataFrame(
    [
        {"granule_id": granule_id, "cell_id": cell_id}
        for granule_id, cell_id in granule_to_cell.items()
    ],
    columns=["granule_id", "cell_id"],
)

# Choose where to save the CSV
output_path = Path("granule_to_cell.csv")

# Save without the pandas row index
assignments.to_csv(output_path, index=False)

print(f"Saved {len(assignments)} assignments to: {output_path.resolve()}")